In [59]:
"""
__authors__ = Marc Vallvé Palerm, Ignasi Rebordosa Cots, Marc Salvatella Ferrerfàbrega
__date__    = Nov 2026
__title__   = Non-Linear Data Structures Project
"""

'\n__authors__ = Marc Vallvé Palerm, Ignasi Rebordosa Cots, Marc Salvatella Ferrerfàbrega\n__date__    = Nov 2026\n__title__   = Non-Linear Data Structures Project\n'

For this exercise, we decided to use an adjacency-list representation. Although a matrix-based representation would also be valid, it is less suitable for a graph with many sites and relatively few routes. For example, with 1,000 sites, a matrix would require a 1,000 × 1,000 structure, which means 1,000,000 cells and O(V²) initialization time, even though most pairs of sites are not connected. With an adjacency list, each site stores only its own data and its outgoing connections, so the structure uses O(V + E) memory, where V is the number of sites and E is the number of routes. Building the graph from the file also takes O(V + E) time on average, since a dictionary maps each site ID to its corresponding site object and allows each route endpoint to be found in O(1) average time. The adjacency list also naturally stores the connection attributes, such as destination, shipping time and restriction, and is more efficient for the traversals, topological sort and Dijkstra algorithm required later. A matrix would still be valid and would provide O(1) edge lookup, but it would waste space on the many non-existing routes.

# 2.1.1 — Data

In [60]:
#Product class
class Site:
    """Stores the data of one production site."""

    def __init__(self, site_id, product, corporation, material):
        self.id = site_id
        self.product = product
        self.corporation = corporation
        self.material = material
        self.connections = []  # List to store connections to other sites

    def get_class_info(self):
        return (
            f"Site(id={self.id}, product={self.product!r}, "
            f"corporation={self.corporation!r}, "
            f"material={self.material})"
        )

class Connection:
    """Stores the data of one connection between two production sites."""

    def __init__(self, origin, destination, shipping_time, restriction):
        self.origin = origin
        self.destination = destination
        self.shipping_time = shipping_time
        self.restriction = restriction

    def get_connection_info(self):
        return (
            f"Connection(origin={self.origin.id}, destination={self.destination.id}, "
            f"shipping_time={self.shipping_time}, restriction={self.restriction})"
        )




# 2.1.2 - Functionalities

### 2.1.2.1 - Graph Representation

In [61]:
# ============================================================
# .paed FILE FORMAT
# ============================================================

# First line: number of sites/vertices
# Number of sites

# <site_id>;<product_name>;<corporation>;<site_material> (*Number of sites)

# Next line: number of routes/edges
# Number of routes

# <origin_id>;<destination_id>;<shipping_time>;<restriction> (*Number of routes)

file_name = r"Datasets/Graphs/aCityXS.paed"

sites = [] #a list to store all the site objects
sites_by_id = {} #stores side id as primary key: site object to access them faster direct access, O(1), instead of serial access, O(n)

with open(file_name, "r") as f:
    #read first line to get number of sites
    num_sites = int(f.readline().strip())

    #initialize a list to store the sites
    for i in range(num_sites):
        #read the line and split it by semicolon
        line = f.readline().strip()
        site_id, product, corporation, material = line.split(";")
        #create a site object and append it to the list
        sites.append(Site(int(site_id), product, corporation, material))
        sites_by_id[int(site_id)] = sites[i] #store the site object in the dictionary with its id as primary key
        
    #get how many routes there are
    num_routes = int(f.readline().strip())

    for i in range(num_routes):
        #read the line and split it by semicolon
        line = f.readline().strip()

        origin_id, destination_id, shipping_time, restriction = line.split(";")
        
        #create a Connection object and append it to the list of connections of the origin site
        #----------get the site where the connection starts and ends
        origin_site = sites_by_id[int(origin_id)]
        destination_site = sites_by_id[int(destination_id)]
        origin_site.connections.append(Connection(origin_site, destination_site, int(shipping_time), restriction))

In [62]:
#represent graph as adjacency list
for site in sites:
    print(f"\nSite {site.id} ({site.product}) has connections:")
    if not site.connections:
        print(f"\t-> No connections found")
    else:
        for connection in site.connections:
            print(f"\t-> Site {connection.destination.id} ({connection.destination.product}), "
                  f"Shipping Time: {connection.shipping_time}, Restriction: {connection.restriction}")



Site 67 (Triangular prism Catun No.2) has connections:
	-> Site 103 (Copper cuboid Mk II), Shipping Time: 19, Restriction: GLASS
	-> Site 118 (Copper hemisphere edge type 3), Shipping Time: 14, Restriction: NONE

Site 87 (Iron tetrahedron Mk I) has connections:
	-> No connections found

Site 119 (Fabric tetrahedron Catun No.1) has connections:
	-> Site 103 (Copper cuboid Mk II), Shipping Time: 24, Restriction: PLASTIC
	-> Site 121 (Wood cone mid-grade salvage), Shipping Time: 13, Restriction: GLASS
	-> Site 78 (Iron hexagonal prism edge type 1), Shipping Time: 22, Restriction: ORGANIC

Site 123 (Tinted glass icosahedron rusted junk) has connections:
	-> Site 103 (Copper cuboid Mk II), Shipping Time: 12, Restriction: PLASTIC

Site 101 (Tinted glass cube Mk I) has connections:
	-> Site 77 (Wax parallelepiped Mk II), Shipping Time: 19, Restriction: GLASS
	-> Site 134 (Polystyrene hexagonal prism Mk II), Shipping Time: 11, Restriction: METAL

Site 77 (Wax parallelepiped Mk II) has connect

A -> Site ID entered by the user.

To retrieve it from our code, we will use sites_by_id[ID].

A corporation can have more than one site.

We will create a tree that starts with all the connections of A. The second level will contain the connections of A, the third level will contain the connections of those connections, and so on.

A
|-B
|_ C
...

To find all nodes reachable from A that belong to the same corporation, we will use DFS.

Since we need to look through all reachable nodes, as there may be more than one correct answer, the time complexity is O(n).

### 2.1.2.2 - Commercial Intelligence

In [64]:
ID_to_search = int(input("Enter the ID of the site to search for: "))

site_to_search = sites_by_id[ID_to_search]

visited = set() # Set to keep track of visited nodes during the search and do not repeat them
stack_corporation_nodes = [] # Stack of the result of the nodes that belong to the same corporation
corporation_to_search = site_to_search.corporation

#initialize a queue for sites waiting to be visited and append initial site (search input)
queue = []
queue.append(site_to_search)

#dict storing id primary key: site's distance from search site (depth)
depth_by_site_id = {site_to_search.id: 0}

# recursive function to find all the nodes that belong to the same corporation
def find_corporation_nodes(queue_index, corporation):

    if queue_index >= len(queue):
        return

    current_site = queue[queue_index]

    for connection in current_site.connections:
        next_site = connection.destination

        #only add site first time reached
        if next_site.id not in visited:
            visited.add(next_site.id)

            depth_by_site_id[next_site.id] = (depth_by_site_id[current_site.id] + 1)
            queue.append(next_site)

            #keep only matching corporation sites, exclude search site
            if next_site.corporation == corporation and next_site.id != ID_to_search:
                stack_corporation_nodes.append(next_site)

    #process next queued site. new sites are appended to queue to continue BFS
    find_corporation_nodes(queue_index + 1, corporation)

visited.add(site_to_search.id)
find_corporation_nodes(0, corporation_to_search)

#sort sites from deepest to shallows, using ascenfing ID for ties
stack_corporation_nodes.sort(key=lambda site: (-depth_by_site_id[site.id], site.id))

#display the result
print(f"Search results:")
if not stack_corporation_nodes:
    print("No other accessible sites found for this corporation.")
else:
    for site in stack_corporation_nodes:
        print(
            f"\tSite {site.id} ({site.product}) — "
            f"Corporation: {site.corporation} — "
            f"Supply-chain depth from site {ID_to_search}: "
            f"{depth_by_site_id[site.id]}"
    )

Search results:
	Site 101 (Tinted glass cube Mk I) — Corporation: Jones Inc — Supply-chain depth from site 119: 2
	Site 121 (Wood cone mid-grade salvage) — Corporation: Jones Inc — Supply-chain depth from site 119: 1
